# MAMA-MIA — 12: Maqola rasmi uchun kesim eksporti

Sifat jihatidan ko'rsatuvchi rasm uchun bir nechta namunaviy kesim
eksport qilinadi. **GPU kerak emas, ~10 daqiqa.**

Faqat tasvir va ground truth olinadi — model vazni kerak emas.
Rasmning vazifasi qanday ma'lumot yo'qolishini ko'rsatish, bu esa
kirishga tegishli.

Har bir manbadan bittadan holat tanlanadi: o'smasi eng katta bo'lgan
kesim, chunki u ko'rsatishga eng mos.

Chiqish: `figure_slices.npz` — bir necha yuz kilobayt.

**Input:** variantlar notebook'i (`V2_iso_scale`).

**Litsenziya:** MAMA-MIA CC BY-NC 4.0. Chiqish datasetini Private
saqlang. Maqolada chop etish mumkin, chunki Procedia maqolalari
CC BY-NC-ND ostida chiqadi — NC sharti bilan mos.

In [ ]:
import os, glob, json
import numpy as np, h5py
from scipy.ndimage import label, binary_erosion

IMG = 192
cand = sorted({os.path.dirname(p) for p in
               glob.glob("/kaggle/input/**/V2_iso_scale/mamamia_*.h5", recursive=True)})
assert cand, "V2_iso_scale topilmadi — variantlar notebook'ini Input qiling"
DATA = cand[0]
print("V2:", DATA)

OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

AREA_MIN, AREA_MAX, TOPK = 150, 1500, 3


def norm(vol):
    """V2 pipeline'dagi bilan bir xil: 1–99 persentil, keyin z-score."""
    lo, hi = np.percentile(vol, [1, 99])
    if hi <= lo:
        lo, hi = float(vol.min()), float(vol.max()) + 1e-3
    c = np.clip(vol, lo, hi).astype(np.float32)
    return (c - c.mean()) / (c.std() + 1e-6)


def shape_score(m):
    """Bitta ixcham o'choqmi? Diffuz/ko'p o'choqli kesimlar rad etiladi.

    Qaytaradi: (maydon, ixchamlik) yoki None.
    Ixchamlik = 4*pi*A/P^2 — disk uchun ~1, sochilgan naqsh uchun kichik.
    """
    if not m.any():
        return None
    _, n = label(m)
    if n != 1:                      # ko'p o'choqli — rasm uchun yaramaydi
        return None
    a = int(m.sum())
    if not (AREA_MIN <= a <= AREA_MAX):
        return None
    per = int((m & ~binary_erosion(m, border_value=0)).sum())
    return a, 4 * np.pi * a / max(per ** 2, 1)


picks = {}
for p in sorted(glob.glob(os.path.join(DATA, "mamamia_*.h5"))):
    src = os.path.basename(p)[len("mamamia_"):-len(".h5")]
    cands = []
    with h5py.File(p, "r") as h:
        for key in list(h)[:150]:
            m3 = h[key]["msk"]
            for z in range(m3.shape[2]):
                sc = shape_score(np.asarray(m3[:, :, z]).astype(bool))
                if sc:
                    cands.append((sc[1], sc[0], key, z))
        if not cands:
            print(f"  {src}: mos kesim topilmadi"); continue
        cands.sort(reverse=True)                     # ixchamlik bo'yicha
        # Har bir HOLATdan faqat bitta kesim: aks holda uchala nomzod
        # bitta o'smaning qo'shni kesimlari bo'lib chiqadi va tanlov bo'lmaydi.
        seen, chosen = set(), []
        for c in cands:
            if c[2] in seen:
                continue
            seen.add(c[2]); chosen.append(c)
            if len(chosen) == TOPK:
                break
        picks[src] = []
        for comp, a, key, z in chosen:
            img = np.asarray(h[key]["img"][:, :, :], dtype=np.float32)
            picks[src].append({
                "key": key, "z": int(z), "area": int(a), "comp": float(comp),
                "img": norm(img)[:, :, z].astype(np.float32),
                "msk": np.asarray(h[key]["msk"][:, :, z]).astype(np.uint8)})
        print(f"  {src:<7} {len(cands):>5} nomzod / {len(seen)} holat → {len(chosen)} ta  "
              + ", ".join(f"{c[2]}/z{c[3]} (A={c[1]}, ixch={c[0]:.2f})"
                          for c in chosen))

arrays = {}
meta = {}
for s, lst in picks.items():
    meta[s] = []
    for i, v in enumerate(lst):
        arrays[f"{s}_{i}_img"] = v["img"]
        arrays[f"{s}_{i}_msk"] = v["msk"]
        meta[s].append({k: v[k] for k in ("key", "z", "area", "comp")})

np.savez_compressed(os.path.join(OUT, "figure_slices.npz"),
                    meta=json.dumps(meta), **arrays)
sz = os.path.getsize(os.path.join(OUT, "figure_slices.npz"))
print(f"\nfigure_slices.npz yozildi — {sz/1024:.0f} KB, "
      f"{len(picks)} manba x {TOPK} nomzod")

## Tekshirish

Eksport qilingan kesimlarni ko'rib chiqamiz — o'sma ko'rinib turibdimi
va kadr ichidami.

In [ ]:
import matplotlib.pyplot as plt

srcs = sorted(picks)
rows, cols = len(srcs), TOPK
fig, axes = plt.subplots(rows, cols, figsize=(3.0 * cols, 3.1 * rows))
axes = np.atleast_2d(axes)
for i, s in enumerate(srcs):
    for j in range(cols):
        ax = axes[i, j]
        if j < len(picks[s]):
            v = picks[s][j]
            ax.imshow(v["img"], cmap="gray")
            ax.contour(v["msk"], levels=[0.5], colors="#eb6834", linewidths=1.2)
            ax.set_title(f"{s} #{j}  A={v['area']}  ixch={v['comp']:.2f}",
                         fontsize=9)
        ax.axis("off")
plt.tight_layout()
plt.show()

print("Har qatordagi uchta nomzoddan qaysi biri eng yaxshi ko'rinsa, raqamini ayting")
print("(masalan: DUKE #1, NACT #0). Hammasi npz ichida — qayta ishga tushirish shart emas.")

## Keyingi qadam

`figure_slices.npz` ni yuklab olib yuboring. Rasmni maqola uslubida
o'zim chizaman:

- bitta kesim 1.5 / 2.0 / 3.0 / 4.55 mm effektiv qadamda — rezolyutsiya
  yo'qolishi ko'z bilan ko'rinadi
- DUKE holatida k=1 va k=2 — o'smaning kadrdan chiqib ketishi